# 03 — Denoising Autoencoder on Fashion-MNIST
### Deep Learning Week 3: Self-Supervised Representation Learning & Image Reconstruction

---

## 1. Conceptual Introduction: Autoencoders and the Denoising Criterion

An **Autoencoder** is an unsupervised / self-supervised neural network trained to reconstruct its input through a constrained bottleneck representation:
$$x \xrightarrow{\text{Encoder } f_\theta} z \xrightarrow{\text{Decoder } g_\phi} \hat{x} \approx x$$

Where:
- $z = f_\theta(x) \in \mathbb{R}^d$ is the compact **latent representation** (bottleneck).
- $\hat{x} = g_\phi(z) \in \mathbb{R}^D$ is the **reconstructed output**.

If an autoencoder simply learned the identity mapping $g(f(x)) = x$, it could memorize inputs without discovering useful semantic features.

### The Denoising Criterion (Vincent et al., 2008)
A **Denoising Autoencoder (DAE)** introduces a stochastic corruption process $q(\tilde{x} \mid x)$ to the input:
$$x \xrightarrow{\text{Corrupt}} \tilde{x} = x + \epsilon \quad (\epsilon \sim \mathcal{N}(0, \sigma^2))$$
$$\tilde{x} \xrightarrow{f_\theta} z \xrightarrow{g_\phi} \hat{x}$$

Crucially, the reconstruction loss penalizes the difference between $\hat{x}$ and the **original clean image $x$**, not the corrupted input $\tilde{x}$:
$$\mathcal{L}(x, \hat{x}) = \|x - g_\phi(f_\theta(\tilde{x}))\|^2$$

This forces the autoencoder to:
1. Learn the underlying **low-dimensional data manifold** $\mathcal{M}$ where realistic clothing images reside.
2. Act as a **projection operator** that maps off-manifold noisy samples back onto $\mathcal{M}$.


In [ ]:
# 2. Imports and Environment Setup
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")


## 3. Dataset Loading & Inspection: Fashion-MNIST

Fashion-MNIST consists of 70,000 $28 \times 28$ grayscale images across 10 clothing categories:
`0: T-shirt/top`, `1: Trouser`, `2: Pullover`, `3: Dress`, `4: Coat`, `5: Sandal`, `6: Shirt`, `7: Sneaker`, `8: Bag`, `9: Ankle boot`.


In [ ]:
# 3.1 Load Fashion-MNIST Dataset
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print(f"Train set: {x_train_raw.shape}, Test set: {x_test_raw.shape}")

# Normalize pixel values to [0.0, 1.0] and add channel dimension
x_train = x_train_raw.astype("float32") / 255.0
x_test = x_test_raw.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)  # Shape: (60000, 28, 28, 1)
x_test = np.expand_dims(x_test, axis=-1)    # Shape: (10000, 28, 28, 1)

print(f"Processed input shape: {x_train.shape}, range: [{x_train.min():.1f}, {x_train.max():.1f}]")


## 4. The Corruption Process: Additive Gaussian Noise

We apply zero-mean additive Gaussian noise with $\sigma = 0.35$, followed by clipping to $[0.0, 1.0]$:
$$\tilde{x} = \text{clip}(x + \mathcal{N}(0, \sigma^2), 0.0, 1.0)$$


In [ ]:
# 4.1 Corrupt Images with Gaussian Noise
NOISE_FACTOR = 0.35

x_train_noisy = x_train + NOISE_FACTOR * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + NOISE_FACTOR * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0)
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0)

# 4.2 Visualize Clean vs. Noisy Images
n_samples = 8
plt.figure(figsize=(16, 4))
for i in range(n_samples):
    # Clean image
    ax = plt.subplot(2, n_samples, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(f"Clean: {CLASS_NAMES[y_train[i]]}", fontsize=10)
    plt.axis("off")

    # Noisy image
    ax = plt.subplot(2, n_samples, i + 1 + n_samples)
    plt.imshow(x_train_noisy[i].squeeze(), cmap="gray")
    plt.title("Noisy Input", fontsize=10)
    plt.axis("off")

plt.suptitle("Fashion-MNIST: Clean Targets vs. Corrupted Inputs", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Convolutional Autoencoder Architecture

We construct a **Convolutional Autoencoder (CAE)**:
- **Encoder:**
  - `Conv2D(32, (3,3), relu, padding='same')` $\to (28, 28, 32)$
  - `MaxPooling2D((2,2))` $\to (14, 14, 32)$
  - `Conv2D(16, (3,3), relu, padding='same')` $\to (14, 14, 16)$
  - `MaxPooling2D((2,2))` $\to (7, 7, 16)$ (Bottleneck: 784 latent variables)
- **Decoder:**
  - `Conv2DTranspose(16, (3,3), strides=2, relu, padding='same')` $\to (14, 14, 16)$
  - `Conv2DTranspose(32, (3,3), strides=2, relu, padding='same')` $\to (28, 28, 32)$
  - `Conv2D(1, (3,3), sigmoid, padding='same')` $\to (28, 28, 1)$ (Clean reconstruction)


In [ ]:
# 5.1 Define Convolutional Denoising Autoencoder
inputs = layers.Input(shape=(28, 28, 1), name="noisy_image_input")

# Encoder
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="enc_conv1")(inputs)
x = layers.MaxPooling2D((2, 2), padding="same", name="enc_pool1")(x)
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same", name="enc_conv2")(x)
encoded = layers.MaxPooling2D((2, 2), padding="same", name="latent_bottleneck")(x)

# Decoder
x = layers.Conv2DTranspose(16, (3, 3), strides=2, activation="relu", padding="same", name="dec_transconv1")(encoded)
x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu", padding="same", name="dec_transconv2")(x)
decoded = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same", name="reconstructed_clean_image")(x)

autoencoder = keras.Model(inputs=inputs, outputs=decoded, name="Fashion_Denoising_Autoencoder")
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
autoencoder.summary()


## 6. Model Training: Reconstructing Clean Targets

We train the autoencoder by feeding `x_train_noisy` as input and targeting `x_train` (clean targets).


In [ ]:
# 6.1 Train the Denoising Autoencoder
EPOCHS = 8
BATCH_SIZE = 128

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
]

history = autoencoder.fit(
    x_train_noisy,
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1,
)


## 7. Reconstruction Evaluation & Loss Curves


In [ ]:
# 7.1 Plot Training and Validation Loss Curves
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Train MSE Loss", marker="o", color="#1f77b4")
plt.plot(history.history["val_loss"], label="Val MSE Loss", marker="s", color="#ff7f0e")
plt.title("Denoising Autoencoder Loss (MSE) Across Epochs", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# Evaluate on Held-Out Test Set
test_mse = autoencoder.evaluate(x_test_noisy, x_test, verbose=0)
print(f"Held-Out Test MSE: {test_mse:.5f}")


## 8. Side-by-Side Visual Comparison: Clean vs. Noisy vs. Denoised

We display an 8-column visual comparison showing:
1. **Row 1:** Ground Truth Clean Images ($x$)
2. **Row 2:** Corrupted Noisy Images ($\tilde{x}$)
3. **Row 3:** Denoised Reconstructions ($\hat{x}$)


In [ ]:
# 8.1 Generate Reconstructions on Test Set
reconstructed_images = autoencoder.predict(x_test_noisy[:n_samples], verbose=0)

plt.figure(figsize=(16, 6))
for i in range(n_samples):
    # 1. Clean
    ax = plt.subplot(3, n_samples, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title(f"Clean: {CLASS_NAMES[y_test[i]]}", fontsize=9)
    plt.axis("off")

    # 2. Noisy
    ax = plt.subplot(3, n_samples, i + 1 + n_samples)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray")
    plt.title("Noisy Input", fontsize=9)
    plt.axis("off")

    # 3. Reconstructed
    ax = plt.subplot(3, n_samples, i + 1 + 2 * n_samples)
    plt.imshow(reconstructed_images[i].squeeze(), cmap="gray")
    plt.title("Denoised Output", fontsize=9)
    plt.axis("off")

plt.suptitle("Denoising Autoencoder: Clean vs. Noisy vs. Reconstructed", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# 8.2 Quantitative Denoising Improvement
mse_noisy = np.mean((x_test[:1000] - x_test_noisy[:1000]) ** 2)
mse_denoised = np.mean((x_test[:1000] - autoencoder.predict(x_test_noisy[:1000], verbose=0)) ** 2)
reduction = (1.0 - mse_denoised / mse_noisy) * 100

print(f"Input Noise MSE:         {mse_noisy:.5f}")
print(f"Post-Denoising MSE:      {mse_denoised:.5f}")
print(f"Noise Error Reduction:   {reduction:.1f}%")


## 9. Observations & Limitations

### What Worked Well:
- The convolutional layers successfully suppress high-frequency uncorrelated Gaussian noise across the background and item silhouettes.
- Structural silhouettes of distinct shapes (e.g., boots, trousers, bags) are accurately preserved.

### Trade-offs and Limitations:
- **Smoothing of Fine Textures:** Because MSE penalizes pixel-by-pixel squared discrepancies, the autoencoder tends to output the average of plausible pixel values. This results in slight blurring of sharp logos, button details, or shoe laces.
- **Capacity Bottleneck:** A bottleneck of $7 \times 7 \times 16 = 784$ features enforces significant compression ($28 \times 28 = 784$ with channels), preventing trivial identity pass-through while retaining salient shape descriptors.
